# 02 – Bivariate Analysis

## Credit Risk – Probability of Default

# 1. Libs

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets

from pathlib import Path
from IPython.display import display, HTML
from IPython.display import display_html
from ucimlrepo import fetch_ucirepo

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

# 2. Data Loading

Loading the Default of Credit Card Clients dataset from the UCI Machine Learning Repository.

In [18]:
df = pd.read_parquet("data/processed/credit_default_eda.parquet")

# 3. Payment Status vs. Credit Default

In [20]:
payment_status_columns = [
    "PAY_1", "PAY_2", "PAY_3",
    "PAY_4", "PAY_5", "PAY_6"
]

tables = []

for col in payment_status_columns:
    result = (
        df.groupby(col)["Y"]
        .agg(["count", "mean"])
        .rename(columns={
            "count": "number_of_clients",
            "mean": "default_rate"
        })
    )

    result["default_rate"] *= 100

    output = widgets.Output(
        layout=widgets.Layout(width="300px")
    )

    with output:
        print(col)
        display(result.round(2))

    tables.append(output)

display(
    widgets.VBox(
        [
            widgets.HBox(tables[:3], layout=widgets.Layout(gap="30px")),
            widgets.HBox(tables[3:], layout=widgets.Layout(gap="30px"))
        ],
        layout=widgets.Layout(gap="30px")
    )
)

# 8. Default Rate Analysis of Negative Bill Amounts

Comparing default rates between clients with negative and non-negative monthly bill statement amounts to investigate whether negative balances are associated with different credit risk profiles.

In [10]:
target = "Y"

bill_columns = [f"BILL_AMT{i}" for i in range(1, 7)]

results = []

for col in bill_columns:

    negative_col = f"{col}_NEGATIVE"
    df[negative_col] = df[col] < 0

    result = (
        df.groupby(negative_col)[target]
        .agg(["count", "mean"])
        .rename(columns={
            "count": "number_of_clients",
            "mean": "default_rate"
        })
    )

    result["default_rate"] *= 100

    results.append(
        f"<h4>{col}</h4>" + result.to_html(float_format=lambda x: f"{x:.2f}")
    )

left_column = "".join(results[:3])
right_column = "".join(results[3:])

html = f"""
<div style="display: flex; gap: 30px;">
    <div style="flex: 1;">{left_column}</div>
    <div style="flex: 1;">{right_column}</div>
</div>
"""

display_html(html, raw=True)

,number_of_clients,default_rate
BILL_AMT1_NEGATIVE,,
False,29410,22.19
True,590,18.47
,number_of_clients,default_rate
BILL_AMT2_NEGATIVE,,
False,29331,22.19
True,669,18.98
,number_of_clients,default_rate
BILL_AMT3_NEGATIVE,,
False,29345,22.17


__We observe a consistent pattern: negative invoice amounts are associated with a lower default rate within this sample.__

__This could be consistent with the hypothesis that these customers hold credit balances in their accounts. However, it remains unclear whether this characteristic truly reflects distinct financial behavior or if it is linked to other variables.__

__We must also take into account that these groups are small, representing approximately 2% of customers in each period.__

__My decision for now: keep the negative values. We have no justification for removing or replacing them.__

__Later, let's also create a feature based on negative bill amounts that could be useful for our model later.__